# H&M Customer Segmentation — Phase 4
## RFM Baseline + Extended Feature Engineering

## Mục tiêu

Phase 4 chuyển dữ liệu sạch từ cấp transaction sang **cấp customer**:

> **1 customer = 1 row**

Phase này thực hiện 6 việc:

1. Chốt định nghĩa R/F/M phù hợp với H&M.
2. Tạo RFM baseline.
3. Tạo candidate Extended features được Phase 3 hỗ trợ.
4. Kiểm tra coverage, variance, skewness và missing sau feature engineering.
5. So sánh mức độ thông tin của các feature mới với RFM.
6. Xuất **master customer feature table** để Phase 5 làm Feature Selection.

> Phase 4 **chưa chọn feature cuối**, chưa chạy K-Means/GMM và chưa dùng ANOVA/MI/Random Forest để lọc.

## 0. Quy ước RFM cho H&M

H&M không có `order_id`, vì vậy:

### Recency

\[
R_i = reference\_date - last\_purchase\_date_i
\]

### Frequency

Không dùng số dòng transaction như số order.

Ta dùng:

\[
F_i = 	ext{number of distinct purchase days}
\]

### Monetary

\[
M_i = \sum price
\]

### Reference date

Dùng:

\[
reference\_date = max(t\_dat) + 1\ day
\]

để mọi Recency đều dương và không phụ thuộc thời điểm chạy notebook.

## 1. Input / Output

Input:

```text
data/processed/
├── transactions_clean.parquet
├── customers_clean.parquet
└── articles_clean.parquet
```

Output:

```text
data/features/
├── customer_rfm_baseline.parquet
├── customer_extended_features.parquet
└── customer_features_master.parquet

reports/
├── figures/
│   └── phase4_features/
└── tables/
    └── phase4_features/
```

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)

PROJECT_ROOT = Path("..")

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
FEATURE_DIR = PROJECT_ROOT / "data" / "features"
FIG_DIR = PROJECT_ROOT / "reports" / "figures" / "phase4_features"
TABLE_DIR = PROJECT_ROOT / "reports" / "tables" / "phase4_features"

for d in [FEATURE_DIR, FIG_DIR, TABLE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

TX_PATH = PROCESSED_DIR / "transactions_clean.parquet"
CUSTOMERS_PATH = PROCESSED_DIR / "customers_clean.parquet"
ARTICLES_PATH = PROCESSED_DIR / "articles_clean.parquet"

for p in [TX_PATH, CUSTOMERS_PATH, ARTICLES_PATH]:
    print(f"{p}: {'OK' if p.exists() else 'MISSING'}")

..\data\processed\transactions_clean.parquet: OK
..\data\processed\customers_clean.parquet: OK
..\data\processed\articles_clean.parquet: OK


## 2. Load dữ liệu

Transaction table lớn nên tiếp tục dùng DuckDB.

Customer/article metadata được đọc bằng Pandas và đăng ký vào DuckDB để join.

In [2]:
# Nếu chưa cài:
# %pip install duckdb pyarrow

import duckdb

con = duckdb.connect()
con.execute("SET max_memory = '3.5GB'")
con.execute("SET temp_directory = '../.duckdb_temp'")
con.execute("SET preserve_insertion_order = false")

customers = pd.read_parquet(CUSTOMERS_PATH)
articles = pd.read_parquet(ARTICLES_PATH)

TX = f"read_parquet('{TX_PATH.as_posix()}')"
ART = f"read_parquet('{ARTICLES_PATH.as_posix()}')"

date_info = con.execute(f'''
SELECT
    MIN(t_dat) AS min_date,
    MAX(t_dat) AS max_date,
    MAX(t_dat) + INTERVAL 1 DAY AS reference_date,
    COUNT(*) AS transaction_rows,
    COUNT(DISTINCT customer_id) AS customers
FROM {TX}
''').df()

display(date_info)
date_info.to_csv(TABLE_DIR / "01_reference_date_and_scope.csv", index=False)

reference_date = pd.to_datetime(date_info.loc[0, "reference_date"])
print("Reference date:", reference_date.date())

,min_date,max_date,reference_date,transaction_rows,customers
0,2018-09-20,2020-09-22,2020-09-23,31788324,1362281


Reference date: 2020-09-23


# A. RFM BASELINE

## 3. Tạo RFM

RFM baseline phải đủ đơn giản để làm **đối chứng** cho mọi Extended feature set sau này.

In [3]:
rfm = con.execute(f'''
WITH base AS (
    SELECT
        customer_id,
        MIN(t_dat) AS first_purchase_date,
        MAX(t_dat) AS last_purchase_date,
        COUNT(DISTINCT t_dat) AS frequency_purchase_days,
        SUM(price) AS monetary_total_spend,
        COUNT(*) AS item_frequency
    FROM {TX}
    GROUP BY customer_id
)
SELECT
    customer_id,
    first_purchase_date,
    last_purchase_date,
    DATE_DIFF(
        'day',
        last_purchase_date,
        DATE '{reference_date.date()}'
    ) AS recency_days,
    frequency_purchase_days,
    monetary_total_spend,
    item_frequency
FROM base
''').df()

display(rfm.head())
print("RFM shape:", rfm.shape)


,customer_id,first_purchase_date,last_purchase_date,recency_days,frequency_purchase_days,monetary_total_spend,item_frequency
0,3d4524f219304eb47422deb88200cb4aec7715dd5985c8...,2018-09-28,2020-07-06,79,12,1.868661,49
1,3fb57f81123110eba102e4f5fef3c7a1d51d2844fd82b3...,2018-09-28,2020-04-11,165,22,1.912051,93
2,4043fdbe22eb397206dabf520d2d88651179724ac19ccf...,2018-09-28,2020-09-18,5,232,8.904407,725
3,407fe7ab0316eb606c6f2245634cb1930fe3e195fd9c67...,2018-09-28,2020-06-08,107,14,0.435424,20
4,40c740a78b527cc0fab5db1db0962015f60bb2b98965cc...,2018-09-28,2019-03-04,569,3,0.335424,10


RFM shape: (1362281, 7)


In [4]:
# Core quality assertions
assert rfm["customer_id"].isna().sum() == 0
assert rfm["customer_id"].duplicated().sum() == 0
assert (rfm["recency_days"] >= 1).all()
assert (rfm["frequency_purchase_days"] >= 1).all()
assert (rfm["monetary_total_spend"] > 0).all()

print("RFM integrity checks: PASSED")

RFM integrity checks: PASSED


In [5]:
rfm_path = FEATURE_DIR / "customer_rfm_baseline.parquet"
rfm.to_parquet(rfm_path, index=False)

rfm_summary = rfm[
    ["recency_days", "frequency_purchase_days", "monetary_total_spend", "item_frequency"]
].describe(percentiles=[.01, .05, .25, .5, .75, .9, .95, .99]).T

display(rfm_summary)
rfm_summary.to_csv(TABLE_DIR / "02_rfm_summary.csv")

,count,mean,std,min,1%,5%,25%,50%,75%,90%,95%,99%,max
recency_days,1362281.0,236.148437,221.118791,1.000000,2.000000,7.000000,49.000000,152.000000,399.000000,609.000000,674.000000,725.000000,734.000000
frequency_purchase_days,1362281.0,6.665423,9.707005,1.000000,1.000000,1.000000,1.000000,3.000000,8.000000,17.000000,24.000000,45.000000,427.000000
monetary_total_spend,1362281.0,0.649386,1.197582,0.000763,0.013542,0.027424,0.088068,0.245593,0.701186,1.622898,2.573542,5.698885,57.676407
item_frequency,1362281.0,23.334631,39.242253,1.000000,1.000000,1.000000,3.000000,9.000000,27.000000,60.000000,91.000000,187.000000,1895.000000


## 4. Trực quan RFM

Ta vẽ cả raw và log-view khi cần để thấy skewness.

Không log-transform dữ liệu model ở Phase 4; chỉ dùng `log1p` để **chẩn đoán**.

In [6]:
rfm_plot_specs = [
    ("recency_days", "Recency (days)", "03_recency_distribution.png"),
    ("frequency_purchase_days", "Frequency (distinct purchase days)", "04_frequency_distribution.png"),
    ("monetary_total_spend", "Monetary (total observed spend)", "05_monetary_distribution.png")
]

for col, label, filename in rfm_plot_specs:
    s = rfm[col].dropna()
    upper = s.quantile(.99)

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.hist(s.clip(upper=upper), bins=50)
    ax.set_xlabel(f"{label} (clipped at p99)")
    ax.set_ylabel("Customers")
    ax.set_title(f"{label} distribution")
    fig.tight_layout()
    fig.savefig(FIG_DIR / filename, dpi=160, bbox_inches="tight")
    plt.show()

C:\Users\Dell\AppData\Local\Temp\ipykernel_14844\1624413107.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


C:\Users\Dell\AppData\Local\Temp\ipykernel_14844\1624413107.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


C:\Users\Dell\AppData\Local\Temp\ipykernel_14844\1624413107.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
for col in ["frequency_purchase_days", "monetary_total_spend"]:
    s = rfm[col].dropna()
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.hist(np.log1p(s), bins=50)
    ax.set_xlabel(f"log1p({col})")
    ax.set_ylabel("Customers")
    ax.set_title(f"{col} — log diagnostic view")
    fig.tight_layout()
    fig.savefig(FIG_DIR / f"06_log_{col}.png", dpi=160, bbox_inches="tight")
    plt.show()

C:\Users\Dell\AppData\Local\Temp\ipykernel_14844\212735742.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


# B. EXTENDED TEMPORAL FEATURES

## 5. Interpurchase Time và Purchase Regularity

Dùng **distinct purchase dates** để tránh multiple items trong cùng ngày tạo interval bằng 0.

Candidate features:

- `mean_interpurchase_days`
- `median_interpurchase_days`
- `std_interpurchase_days`
- `interval_count`

Lưu ý:

- Mean/median cần ít nhất 2 purchase days.
- Standard deviation cần ít nhất 3 purchase days để có ít nhất 2 intervals.

Missing ở đây là **structural missing**, không phải lỗi dữ liệu.

In [8]:
temporal = con.execute(f'''
WITH days AS (
    SELECT DISTINCT customer_id, t_dat
    FROM {TX}
),
ordered AS (
    SELECT
        customer_id,
        t_dat,
        LAG(t_dat) OVER (
            PARTITION BY customer_id
            ORDER BY t_dat
        ) AS prev_date
    FROM days
),
diffs AS (
    SELECT
        customer_id,
        DATE_DIFF('day', prev_date, t_dat) AS interval_days
    FROM ordered
    WHERE prev_date IS NOT NULL
)
SELECT
    customer_id,
    COUNT(*) AS interval_count,
    AVG(interval_days) AS mean_interpurchase_days,
    MEDIAN(interval_days) AS median_interpurchase_days,
    STDDEV_SAMP(interval_days) AS std_interpurchase_days,
    MIN(interval_days) AS min_interpurchase_days,
    MAX(interval_days) AS max_interpurchase_days
FROM diffs
GROUP BY customer_id
''').df()

display(temporal.head())

,customer_id,interval_count,mean_interpurchase_days,median_interpurchase_days,std_interpurchase_days,min_interpurchase_days,max_interpurchase_days
0,0002e6a17e3a75600888cac019846a600c9c6fe2bc1565...,6,81.833333,65.5,82.022964,10,235
1,00061f9d8a57529d6a662d6ca9f92c82f08a2fe5de53b4...,1,42.000000,42.0,NaN,42,42
2,0008968c0d451dbc5a9968da03196fe20051965edde741...,36,20.138889,13.5,20.685610,1,88
3,0008f66f5804877333ab9942c880b4a222f5b1a9f6ce27...,1,72.000000,72.0,NaN,72,72
4,000b1981787a5d7c5353b6f4dd094442f3fc8e4f948e7f...,2,94.500000,94.5,7.778175,89,100


In [9]:
temporal_coverage = pd.DataFrame({
    "feature": [
        "mean_interpurchase_days",
        "median_interpurchase_days",
        "std_interpurchase_days"
    ],
    "coverage_pct": [
        temporal["mean_interpurchase_days"].notna().sum() / len(rfm) * 100,
        temporal["median_interpurchase_days"].notna().sum() / len(rfm) * 100,
        temporal["std_interpurchase_days"].notna().sum() / len(rfm) * 100
    ]
})

display(temporal_coverage)
temporal_coverage.to_csv(TABLE_DIR / "07_temporal_feature_coverage.csv", index=False)

,feature,coverage_pct
0,mean_interpurchase_days,67.115962
1,median_interpurchase_days,67.115962
2,std_interpurchase_days,53.674095


# C. SPENDING / ACTIVITY FEATURES

## 6. Avg Item Price và Avg Spend per Purchase Day

Không dùng tên `AOV` vì dataset không có order_id.

Ta tạo:

\[
AvgItemPrice = rac{Monetary}{ItemFrequency}
\]

\[
AvgSpendPerPurchaseDay = rac{Monetary}{Frequency_{day}}
\]

In [10]:
spending = rfm[[
    "customer_id",
    "monetary_total_spend",
    "item_frequency",
    "frequency_purchase_days"
]].copy()

spending["avg_item_price"] = (
    spending["monetary_total_spend"] /
    spending["item_frequency"]
)

spending["avg_spend_per_purchase_day"] = (
    spending["monetary_total_spend"] /
    spending["frequency_purchase_days"]
)

display(spending.head())

,customer_id,monetary_total_spend,item_frequency,frequency_purchase_days,avg_item_price,avg_spend_per_purchase_day
0,3d4524f219304eb47422deb88200cb4aec7715dd5985c8...,1.868661,49,12,0.038136,0.155722
1,3fb57f81123110eba102e4f5fef3c7a1d51d2844fd82b3...,1.912051,93,22,0.020560,0.086911
2,4043fdbe22eb397206dabf520d2d88651179724ac19ccf...,8.904407,725,232,0.012282,0.038381
3,407fe7ab0316eb606c6f2245634cb1930fe3e195fd9c67...,0.435424,20,14,0.021771,0.031102
4,40c740a78b527cc0fab5db1db0962015f60bb2b98965cc...,0.335424,10,3,0.033542,0.111808


# D. FASHION DIVERSITY FEATURES

## 7. Product / Product-Type / Garment / Colour / Department Diversity

Các feature diversity đo số lượng category khác nhau mà một khách đã mua.

Chúng có thể phản ánh:
- breadth of interest;
- cross-category shopping;
- potential cross-sell opportunity.

Nhưng các feature diversity có thể tương quan rất mạnh với Frequency/ItemFrequency.  
Phase 5 sẽ kiểm tra redundancy trước khi chọn.

In [11]:
# Tối ưu hóa: Thay thế bảng tạm in-memory bằng 2 truy vấn quét native Parquet trực tiếp
# Loại bỏ hoàn toàn lỗi tràn bộ nhớ 'Allocation failure' và tràn đĩa .tmp

div1 = con.execute(f'''
SELECT
    tx.customer_id,
    COUNT(DISTINCT tx.article_id) AS product_diversity,
    COUNT(DISTINCT a.garment_group_no) AS garment_group_diversity,
    COUNT(DISTINCT a.colour_group_code) AS colour_diversity
FROM {TX} tx
JOIN {ART} a USING (article_id)
GROUP BY tx.customer_id
''').df()

div2 = con.execute(f'''
SELECT
    tx.customer_id,
    COUNT(DISTINCT a.product_type_no) AS product_type_diversity,
    COUNT(DISTINCT a.department_no) AS department_diversity,
    COUNT(DISTINCT a.section_no) AS section_diversity
FROM {TX} tx
JOIN {ART} a USING (article_id)
GROUP BY tx.customer_id
''').df()

fashion = div1.merge(div2, on="customer_id")
print("Fashion diversity completed successfully:", fashion.shape)
display(fashion.head())


Fashion diversity completed successfully: (1362281, 7)


,customer_id,product_diversity,garment_group_diversity,colour_diversity,product_type_diversity,department_diversity,section_diversity
0,7d9f16011f1961f356d87afec8e984bdedc21132a414f5...,40,12,12,10,19,11
1,8a86c87ccc9b99dbda50e732c390c6ef370b523e41ba48...,15,10,9,10,13,10
2,8b1d40d6c95c0050041f7b17ab9f40152863637af690c2...,138,16,24,28,51,20
3,8f3a8ffa7738cb7f8618e384d56fe46f406cc05061cfb7...,75,14,25,21,32,21
4,a05eed4e7c05064c49a47290ce288b2dbb1cd4d9eaab14...,1,1,1,1,1,1


# E. SALES CHANNEL FEATURES

## 8. Channel Diversity + Dominant Channel Share

Không gán tên business cho `sales_channel_id` nếu chưa có mapping được xác minh.

Ta chỉ tạo:

- `channel_diversity`
- `dominant_channel_share`

Trong đó:

\[
DominantChannelShare=
rac{\max(count\ per\ channel)}{total\ transaction\ rows}
\]

In [12]:
channel = con.execute(f'''
WITH channel_counts AS (
    SELECT
        customer_id,
        sales_channel_id,
        COUNT(*) AS n
    FROM {TX}
    GROUP BY customer_id, sales_channel_id
),
agg AS (
    SELECT
        customer_id,
        COUNT(*) AS channel_diversity,
        SUM(n) AS total_rows,
        MAX(n) AS dominant_rows
    FROM channel_counts
    GROUP BY customer_id
)
SELECT
    customer_id,
    channel_diversity,
    dominant_rows * 1.0 / total_rows AS dominant_channel_share
FROM agg
''').df()

display(channel.head())

,customer_id,channel_diversity,dominant_channel_share
0,02861f9e8c47fe62384a27b48c62778a20a9d580625177...,1,1.000000
1,1178188bddd8ed07149abfe1fe2f7919de517c028dcae0...,2,0.930283
2,ea7f2a411eec74eb7cb965a4f07150f463cfec416b796e...,1,1.000000
3,0ae5bf8563ebbe76de25d63d2d57a81f471c65df1f0ef4...,2,0.914591
4,5de9e3f15bd2c63b44b73a78fd65948566f602aeea7b14...,2,0.705882


# F. CUSTOMER METADATA

## 9. Metadata dùng làm candidate/profiling

Ta nối:

- `age`
- `club_member_status`
- `fashion_news_frequency`
- `FN`
- `Active`

Nhưng **không mặc định đưa demographic vào clustering**.

Phase 5 sẽ quyết định:
- clustering feature;
- profile-only variable;
- hoặc loại khỏi pipeline.

In [13]:
customer_meta_cols = ["customer_id"]

for c in [
    "age",
    "club_member_status",
    "fashion_news_frequency",
    "FN",
    "Active"
]:
    if c in customers.columns:
        customer_meta_cols.append(c)

customer_meta = customers[customer_meta_cols].copy()
display(customer_meta.head())

,customer_id,age,club_member_status,fashion_news_frequency,FN,Active
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,49.0,ACTIVE,NONE,NaN,NaN
1,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,25.0,ACTIVE,NONE,NaN,NaN
2,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,24.0,ACTIVE,NONE,NaN,NaN
3,00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2...,54.0,ACTIVE,NONE,NaN,NaN
4,00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...,52.0,ACTIVE,REGULARLY,1.0,1.0


# G. MASTER FEATURE TABLE

## 10. Join tất cả candidate features

Mọi join phải là **left join từ RFM customer universe**.

Ta không được làm mất customer chỉ vì một extended feature không tính được.

In [14]:
master = (
    rfm
    .merge(temporal, on="customer_id", how="left")
    .merge(
        spending[[
            "customer_id",
            "avg_item_price",
            "avg_spend_per_purchase_day"
        ]],
        on="customer_id",
        how="left"
    )
    .merge(fashion, on="customer_id", how="left")
    .merge(channel, on="customer_id", how="left")
    .merge(customer_meta, on="customer_id", how="left")
)

print("Master shape:", master.shape)
display(master.head())

Master shape: (1362281, 28)


,customer_id,first_purchase_date,last_purchase_date,recency_days,frequency_purchase_days,monetary_total_spend,item_frequency,interval_count,mean_interpurchase_days,median_interpurchase_days,std_interpurchase_days,min_interpurchase_days,max_interpurchase_days,avg_item_price,avg_spend_per_purchase_day,product_diversity,garment_group_diversity,colour_diversity,product_type_diversity,department_diversity,section_diversity,channel_diversity,dominant_channel_share,age,club_member_status,fashion_news_frequency,FN,Active
0,3d4524f219304eb47422deb88200cb4aec7715dd5985c8...,2018-09-28,2020-07-06,79,12,1.868661,49,11.0,58.818182,28.0,77.841914,1.0,264.0,0.038136,0.155722,41,13,11,19,25,15,2,0.510204,25.0,ACTIVE,REGULARLY,1.0,1.0
1,3fb57f81123110eba102e4f5fef3c7a1d51d2844fd82b3...,2018-09-28,2020-04-11,165,22,1.912051,93,21.0,26.714286,24.0,18.711341,1.0,66.0,0.020560,0.086911,82,15,18,25,38,16,2,0.698925,21.0,ACTIVE,REGULARLY,1.0,1.0
2,4043fdbe22eb397206dabf520d2d88651179724ac19ccf...,2018-09-28,2020-09-18,5,232,8.904407,725,231.0,3.121212,2.0,5.213947,1.0,65.0,0.012282,0.038381,561,19,35,37,85,33,2,0.939310,30.0,ACTIVE,NONE,NaN,NaN
3,407fe7ab0316eb606c6f2245634cb1930fe3e195fd9c67...,2018-09-28,2020-06-08,107,14,0.435424,20,13.0,47.615385,18.0,68.508562,2.0,241.0,0.021771,0.031102,20,12,11,16,17,10,2,0.800000,47.0,ACTIVE,NONE,NaN,NaN
4,40c740a78b527cc0fab5db1db0962015f60bb2b98965cc...,2018-09-28,2019-03-04,569,3,0.335424,10,2.0,78.500000,78.5,106.773124,3.0,154.0,0.033542,0.111808,10,6,4,6,7,7,1,1.000000,23.0,ACTIVE,NONE,NaN,NaN


In [15]:
assert len(master) == len(rfm)
assert master["customer_id"].duplicated().sum() == 0
assert master["customer_id"].isna().sum() == 0

print("Master customer universe preserved: PASSED")

Master customer universe preserved: PASSED


## 11. Thêm derived diagnostic features

Các biến sau hỗ trợ Feature Selection và profiling:

- `customer_tenure_days`
- `items_per_purchase_day`

Chúng **chưa chắc được đưa vào clustering**.

In [16]:
master["first_purchase_date"] = pd.to_datetime(master["first_purchase_date"])
master["last_purchase_date"] = pd.to_datetime(master["last_purchase_date"])

master["customer_tenure_days"] = (
    master["last_purchase_date"] - master["first_purchase_date"]
).dt.days

master["items_per_purchase_day"] = (
    master["item_frequency"] /
    master["frequency_purchase_days"]
)

display(master[[
    "customer_id",
    "customer_tenure_days",
    "items_per_purchase_day"
]].head())

,customer_id,customer_tenure_days,items_per_purchase_day
0,3d4524f219304eb47422deb88200cb4aec7715dd5985c8...,647,4.083333
1,3fb57f81123110eba102e4f5fef3c7a1d51d2844fd82b3...,561,4.227273
2,4043fdbe22eb397206dabf520d2d88651179724ac19ccf...,721,3.125000
3,407fe7ab0316eb606c6f2245634cb1930fe3e195fd9c67...,619,1.428571
4,40c740a78b527cc0fab5db1db0962015f60bb2b98965cc...,157,3.333333


# H. FEATURE QUALITY AFTER ENGINEERING

## 12. Coverage / missing / variance / cardinality

Đây là bảng nền cho Phase 5.

Một feature có thể bị loại sau này nếu:
- coverage quá thấp;
- gần như không biến thiên;
- cực kỳ redundant;
- không có ý nghĩa business;
- không cải thiện clustering.

In [17]:
numeric_candidates = master.select_dtypes(include=[np.number]).columns.tolist()

quality_rows = []

for c in numeric_candidates:
    s = master[c]
    non_null = s.dropna()

    quality_rows.append({
        "feature": c,
        "dtype": str(s.dtype),
        "coverage_pct": s.notna().mean() * 100,
        "missing_pct": s.isna().mean() * 100,
        "n_unique": s.nunique(dropna=True),
        "mean": non_null.mean() if len(non_null) else np.nan,
        "std": non_null.std() if len(non_null) else np.nan,
        "min": non_null.min() if len(non_null) else np.nan,
        "median": non_null.median() if len(non_null) else np.nan,
        "max": non_null.max() if len(non_null) else np.nan,
        "zero_pct": (non_null == 0).mean() * 100 if len(non_null) else np.nan,
        "skewness": non_null.skew() if len(non_null) > 2 else np.nan
    })

feature_quality = pd.DataFrame(quality_rows).sort_values(
    ["coverage_pct", "feature"],
    ascending=[False, True]
)

display(feature_quality)
feature_quality.to_csv(TABLE_DIR / "08_feature_quality_after_engineering.csv", index=False)

,feature,dtype,coverage_pct,missing_pct,n_unique,mean,std,min,median,max,zero_pct,skewness
10,avg_item_price,float64,100.000000,0.000000,408393,0.028755,0.014293,0.000763,0.026133,0.506780,0.000000,4.197033
11,avg_spend_per_purchase_day,float64,100.000000,0.000000,482745,0.096004,0.085696,0.000763,0.072797,22.479763,0.000000,17.630005
18,channel_diversity,int64,100.000000,0.000000,2,1.354740,0.478435,1.000000,1.000000,2.000000,0.000000,0.607233
14,colour_diversity,int64,100.000000,0.000000,45,7.205796,6.405216,1.000000,5.000000,45.000000,0.000000,1.331189
23,customer_tenure_days,int64,100.000000,0.000000,734,273.390732,269.962024,0.000000,206.000000,733.000000,32.884038,0.339288
16,department_diversity,int64,100.000000,0.000000,125,10.039228,11.329325,1.000000,6.000000,165.000000,0.000000,2.051710
19,dominant_channel_share,float64,100.000000,0.000000,9569,0.911284,0.146257,0.500000,1.000000,1.000000,0.000000,-1.458074
1,frequency_purchase_days,int64,100.000000,0.000000,229,6.665423,9.707005,1.000000,3.000000,427.000000,0.000000,4.495763
13,garment_group_diversity,int64,100.000000,0.000000,21,5.684182,4.431841,1.000000,4.000000,21.000000,0.000000,0.844085
3,item_frequency,int64,100.000000,0.000000,747,23.334631,39.242253,1.000000,9.000000,1895.000000,0.000000,5.243189


In [18]:
fig_df = feature_quality.sort_values("coverage_pct")

fig, ax = plt.subplots(figsize=(10, max(6, len(fig_df) * 0.3)))
ax.barh(fig_df["feature"], fig_df["coverage_pct"])
ax.set_xlabel("Coverage (%)")
ax.set_title("Candidate feature coverage after engineering")
fig.tight_layout()
fig.savefig(FIG_DIR / "07_feature_coverage.png", dpi=160, bbox_inches="tight")
plt.show()

C:\Users\Dell\AppData\Local\Temp\ipykernel_14844\4088130587.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 13. Distribution diagnostics cho Extended features

Không cần vẽ tất cả.  
Ta ưu tiên những feature sẽ có khả năng vào Phase 5.

In [19]:
priority_features = [
    "recency_days",
    "frequency_purchase_days",
    "monetary_total_spend",
    "mean_interpurchase_days",
    "std_interpurchase_days",
    "avg_item_price",
    "avg_spend_per_purchase_day",
    "product_diversity",
    "product_type_diversity",
    "garment_group_diversity",
    "colour_diversity",
    "dominant_channel_share",
    "items_per_purchase_day"
]

priority_features = [
    c for c in priority_features
    if c in master.columns
]

for c in priority_features:
    s = master[c].dropna()

    if len(s) == 0:
        continue

    upper = s.quantile(.99)

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.hist(s.clip(upper=upper), bins=50)
    ax.set_xlabel(f"{c} (clipped at p99)")
    ax.set_ylabel("Customers")
    ax.set_title(f"{c} distribution")
    fig.tight_layout()
    fig.savefig(FIG_DIR / f"08_{c}_distribution.png", dpi=160, bbox_inches="tight")
    plt.show()

C:\Users\Dell\AppData\Local\Temp\ipykernel_14844\2997514942.py:37: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


C:\Users\Dell\AppData\Local\Temp\ipykernel_14844\2997514942.py:37: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


C:\Users\Dell\AppData\Local\Temp\ipykernel_14844\2997514942.py:37: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


C:\Users\Dell\AppData\Local\Temp\ipykernel_14844\2997514942.py:37: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


C:\Users\Dell\AppData\Local\Temp\ipykernel_14844\2997514942.py:37: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


C:\Users\Dell\AppData\Local\Temp\ipykernel_14844\2997514942.py:37: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


... (3 outputs left out)

C:\Users\Dell\AppData\Local\Temp\ipykernel_14844\2997514942.py:37: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


C:\Users\Dell\AppData\Local\Temp\ipykernel_14844\2997514942.py:37: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


C:\Users\Dell\AppData\Local\Temp\ipykernel_14844\2997514942.py:37: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


C:\Users\Dell\AppData\Local\Temp\ipykernel_14844\2997514942.py:37: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


# I. REDUNDANCY DIAGNOSTIC

## 14. Spearman correlation

Phase 4 chỉ ghi nhận redundancy tiềm năng.

Ví dụ:
- `item_frequency` có thể gần với `frequency_purchase_days`;
- diversity có thể tăng tự nhiên theo số lần mua;
- Monetary có thể liên hệ mạnh với Frequency.

Không loại feature chỉ vì correlation cao ở Phase 4.

In [20]:
corr_features = [
    c for c in priority_features
    if pd.api.types.is_numeric_dtype(master[c])
]

spearman_corr = master[corr_features].corr(method="spearman")

display(spearman_corr.round(3))
spearman_corr.to_csv(TABLE_DIR / "09_candidate_feature_spearman_correlation.csv")

,recency_days,frequency_purchase_days,monetary_total_spend,mean_interpurchase_days,std_interpurchase_days,avg_item_price,avg_spend_per_purchase_day,product_diversity,product_type_diversity,garment_group_diversity,colour_diversity,dominant_channel_share,items_per_purchase_day
recency_days,1.000,-0.615,-0.541,0.075,0.075,-0.008,-0.077,-0.556,-0.554,-0.546,-0.541,0.410,-0.094
frequency_purchase_days,-0.615,1.000,0.871,-0.425,-0.447,0.004,0.125,0.890,0.881,0.878,0.860,-0.627,0.153
monetary_total_spend,-0.541,0.871,1.000,-0.350,-0.367,0.205,0.573,0.947,0.904,0.903,0.910,-0.526,0.511
mean_interpurchase_days,0.075,-0.425,-0.350,1.000,0.823,-0.002,0.011,-0.356,-0.337,-0.335,-0.342,0.130,0.013
std_interpurchase_days,0.075,-0.447,-0.367,0.823,1.000,-0.005,-0.009,-0.395,-0.384,-0.378,-0.375,0.096,-0.013
avg_item_price,-0.008,0.004,0.205,-0.002,-0.005,1.000,0.426,-0.057,-0.063,-0.018,-0.052,0.039,-0.116
avg_spend_per_purchase_day,-0.077,0.125,0.573,0.011,-0.009,0.426,1.000,0.438,0.367,0.370,0.411,-0.027,0.810
product_diversity,-0.556,0.890,0.947,-0.356,-0.395,-0.057,0.438,1.000,0.960,0.946,0.963,-0.554,0.541
product_type_diversity,-0.554,0.881,0.904,-0.337,-0.384,-0.063,0.367,0.960,1.000,0.964,0.927,-0.570,0.464
garment_group_diversity,-0.546,0.878,0.903,-0.335,-0.378,-0.018,0.370,0.946,0.964,1.000,0.924,-0.566,0.441


In [21]:
fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(spearman_corr.values, aspect="auto", vmin=-1, vmax=1)
ax.set_xticks(range(len(spearman_corr.columns)))
ax.set_xticklabels(spearman_corr.columns, rotation=90)
ax.set_yticks(range(len(spearman_corr.index)))
ax.set_yticklabels(spearman_corr.index)
ax.set_title("Candidate feature Spearman correlation")
fig.colorbar(im, ax=ax, label="Spearman correlation")
fig.tight_layout()
fig.savefig(FIG_DIR / "09_candidate_feature_correlation.png", dpi=160, bbox_inches="tight")
plt.show()

C:\Users\Dell\AppData\Local\Temp\ipykernel_14844\3613338675.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


# J. FEATURE DICTIONARY

## 15. Tài liệu hóa công thức và vai trò

Bảng này rất quan trọng cho báo cáo vì nó trả lời:

> “Feature được tạo từ đâu và đại diện hành vi gì?”

In [22]:
feature_dictionary_rows = [
    ["recency_days", "RFM", "reference_date - last_purchase_date", "How recently the customer purchased", "CORE"],
    ["frequency_purchase_days", "RFM", "COUNT(DISTINCT t_dat)", "Purchase frequency using purchase days", "CORE"],
    ["monetary_total_spend", "RFM", "SUM(price)", "Observed cumulative customer spending", "CORE"],
    ["item_frequency", "Activity", "COUNT(transaction rows)", "Number of purchased item rows, not orders", "CANDIDATE"],
    ["mean_interpurchase_days", "Temporal", "MEAN(diff distinct purchase dates)", "Typical repurchase interval", "CANDIDATE"],
    ["median_interpurchase_days", "Temporal", "MEDIAN(diff distinct purchase dates)", "Robust typical repurchase interval", "CANDIDATE"],
    ["std_interpurchase_days", "Temporal", "STD(diff distinct purchase dates)", "Purchase regularity/irregularity", "CANDIDATE"],
    ["avg_item_price", "Spending", "Monetary / item_frequency", "Typical purchased item price", "CANDIDATE"],
    ["avg_spend_per_purchase_day", "Spending", "Monetary / Frequency", "Typical spend on an active purchase day", "CANDIDATE"],
    ["product_diversity", "Fashion", "COUNT(DISTINCT article_id)", "Breadth of individual products purchased", "CANDIDATE"],
    ["product_type_diversity", "Fashion", "COUNT(DISTINCT product_type)", "Breadth across product types", "CANDIDATE"],
    ["garment_group_diversity", "Fashion", "COUNT(DISTINCT garment_group)", "Breadth across garment groups", "CANDIDATE"],
    ["colour_diversity", "Fashion", "COUNT(DISTINCT colour_group)", "Breadth across colour groups", "CANDIDATE"],
    ["department_diversity", "Fashion", "COUNT(DISTINCT department)", "Breadth across departments", "CANDIDATE"],
    ["section_diversity", "Fashion", "COUNT(DISTINCT section)", "Breadth across sections", "CANDIDATE"],
    ["channel_diversity", "Channel", "COUNT(DISTINCT sales_channel_id)", "Number of channels used", "CANDIDATE"],
    ["dominant_channel_share", "Channel", "max(channel transactions)/all transactions", "Strength of preference for one channel", "CANDIDATE"],
    ["customer_tenure_days", "Temporal", "last_purchase_date - first_purchase_date", "Observed customer relationship span", "DIAGNOSTIC"],
    ["items_per_purchase_day", "Activity", "item_frequency / Frequency", "Shopping intensity per active day", "CANDIDATE"],
    ["age", "Metadata", "customers.age", "Customer age", "PROFILE_OR_CANDIDATE"],
    ["club_member_status", "Metadata", "customers.club_member_status", "Membership status", "PROFILE_ONLY_FIRST"],
    ["fashion_news_frequency", "Metadata", "customers.fashion_news_frequency", "Fashion-news engagement metadata", "PROFILE_ONLY_FIRST"]
]

feature_dictionary = pd.DataFrame(
    feature_dictionary_rows,
    columns=[
        "feature",
        "group",
        "formula_or_source",
        "business_interpretation",
        "phase4_role"
    ]
)

# Giữ lại những feature thực sự tồn tại hoặc là metadata được nêu rõ.
feature_dictionary["exists_in_master"] = feature_dictionary["feature"].isin(master.columns)

display(feature_dictionary)
feature_dictionary.to_csv(TABLE_DIR / "10_feature_dictionary.csv", index=False)

,feature,group,formula_or_source,business_interpretation,phase4_role,exists_in_master
0,recency_days,RFM,reference_date - last_purchase_date,How recently the customer purchased,CORE,True
1,frequency_purchase_days,RFM,COUNT(DISTINCT t_dat),Purchase frequency using purchase days,CORE,True
2,monetary_total_spend,RFM,SUM(price),Observed cumulative customer spending,CORE,True
3,item_frequency,Activity,COUNT(transaction rows),"Number of purchased item rows, not orders",CANDIDATE,True
4,mean_interpurchase_days,Temporal,MEAN(diff distinct purchase dates),Typical repurchase interval,CANDIDATE,True
5,median_interpurchase_days,Temporal,MEDIAN(diff distinct purchase dates),Robust typical repurchase interval,CANDIDATE,True
6,std_interpurchase_days,Temporal,STD(diff distinct purchase dates),Purchase regularity/irregularity,CANDIDATE,True
7,avg_item_price,Spending,Monetary / item_frequency,Typical purchased item price,CANDIDATE,True
8,avg_spend_per_purchase_day,Spending,Monetary / Frequency,Typical spend on an active purchase day,CANDIDATE,True
9,product_diversity,Fashion,COUNT(DISTINCT article_id),Breadth of individual products purchased,CANDIDATE,True


# K. CANDIDATE FEATURE POOL CHO PHASE 5

## 16. Không chọn final set — chỉ chốt candidate pool

Nguyên tắc:

### Baseline bắt buộc
- Recency
- Frequency
- Monetary

### Extended candidates
Ưu tiên behavior trước demographic.

Phase 5 sẽ dùng:
1. quality filter;
2. redundancy;
3. Filter Method;
4. Embedded Method;
5. single-addition experiments;
6. forward combination;
7. ablation.

để quyết định feature cuối.

In [23]:
baseline_features = [
    "recency_days",
    "frequency_purchase_days",
    "monetary_total_spend"
]

extended_candidates = [
    "mean_interpurchase_days",
    "std_interpurchase_days",
    "avg_item_price",
    "avg_spend_per_purchase_day",
    "product_diversity",
    "product_type_diversity",
    "garment_group_diversity",
    "colour_diversity",
    "dominant_channel_share",
    "items_per_purchase_day"
]

extended_candidates = [
    c for c in extended_candidates
    if c in master.columns
]

candidate_pool = pd.DataFrame({
    "feature": baseline_features + extended_candidates,
    "role": (
        ["BASELINE"] * len(baseline_features) +
        ["EXTENDED_CANDIDATE"] * len(extended_candidates)
    )
})

candidate_pool = candidate_pool.merge(
    feature_quality[[
        "feature",
        "coverage_pct",
        "missing_pct",
        "n_unique",
        "skewness"
    ]],
    on="feature",
    how="left"
)

display(candidate_pool)
candidate_pool.to_csv(TABLE_DIR / "11_phase5_candidate_feature_pool.csv", index=False)

,feature,role,coverage_pct,missing_pct,n_unique,skewness
0,recency_days,BASELINE,100.000000,0.000000,734,0.801584
1,frequency_purchase_days,BASELINE,100.000000,0.000000,229,4.495763
2,monetary_total_spend,BASELINE,100.000000,0.000000,322671,6.276877
3,mean_interpurchase_days,EXTENDED_CANDIDATE,67.115962,32.884038,15753,2.655421
4,std_interpurchase_days,EXTENDED_CANDIDATE,53.674095,46.325905,489657,2.191751
5,avg_item_price,EXTENDED_CANDIDATE,100.000000,0.000000,408393,4.197033
6,avg_spend_per_purchase_day,EXTENDED_CANDIDATE,100.000000,0.000000,482745,17.630005
7,product_diversity,EXTENDED_CANDIDATE,100.000000,0.000000,585,4.576973
8,product_type_diversity,EXTENDED_CANDIDATE,100.000000,0.000000,63,1.456046
9,garment_group_diversity,EXTENDED_CANDIDATE,100.000000,0.000000,21,0.844085


# L. EXPORT MASTER TABLE

## 17. Lưu dữ liệu cho Phase 5

Không fill missing temporal feature ở đây.

Phase 5 sẽ quyết định:
- impute thế nào;
- dùng missing indicator hay không;
- hay loại feature vì coverage không đủ.

In [24]:
RFM_PATH = FEATURE_DIR / "customer_rfm_baseline.parquet"
EXT_PATH = FEATURE_DIR / "customer_extended_features.parquet"
MASTER_PATH = FEATURE_DIR / "customer_features_master.parquet"

rfm.to_parquet(RFM_PATH, index=False)

extended_cols = [
    "customer_id"
] + [
    c for c in master.columns
    if c not in rfm.columns
]

master[extended_cols].to_parquet(EXT_PATH, index=False)
master.to_parquet(MASTER_PATH, index=False)

print("Saved:", RFM_PATH)
print("Saved:", EXT_PATH)
print("Saved:", MASTER_PATH)

Saved: ..\data\features\customer_rfm_baseline.parquet
Saved: ..\data\features\customer_extended_features.parquet
Saved: ..\data\features\customer_features_master.parquet


# M. POST-EXPORT VALIDATION

## 18. Validation cuối Phase 4

Điều kiện:

- 1 customer = 1 row;
- số customer master = số customer RFM;
- R/F/M không missing;
- Monetary > 0;
- Frequency >= 1;
- Recency >= 1;
- extended missing được giữ nguyên có chủ đích.

In [25]:
master_check = pd.read_parquet(MASTER_PATH)

assert master_check["customer_id"].duplicated().sum() == 0
assert len(master_check) == len(rfm)

for c in [
    "recency_days",
    "frequency_purchase_days",
    "monetary_total_spend"
]:
    assert master_check[c].isna().sum() == 0

assert (master_check["recency_days"] >= 1).all()
assert (master_check["frequency_purchase_days"] >= 1).all()
assert (master_check["monetary_total_spend"] > 0).all()

print("Phase 4 master feature table validation: PASSED")

Phase 4 master feature table validation: PASSED


# N. REPORT-READY CONCLUSION TEMPLATE

## 19. Kết luận Phase 4

Sau khi chạy notebook, điền số thật:

> RFM baseline được xây dựng ở cấp customer với `Recency` tính từ ngày mua cuối cùng đến ngày tham chiếu ..., `Frequency` được định nghĩa là số ngày mua hàng khác nhau thay vì số order do H&M không cung cấp `order_id`, và `Monetary` là tổng giá trị `price` quan sát được.
>
> Ngoài ba đặc trưng RFM, nghiên cứu tạo thêm các candidate feature thuộc bốn nhóm: temporal behavior, spending behavior, fashion diversity và channel behavior. Các temporal features có coverage ...%, phản ánh việc chỉ những customer có mua lặp lại mới có thể tính Interpurchase Time/Regularity.
>
> Phân tích phân bố cho thấy các biến ... có skewness cao, vì vậy Phase Transformation sẽ kiểm tra log transformation và scaling thay vì loại bỏ high-value customers.
>
> Correlation diagnostic cho thấy các cặp ... có mức tương quan mạnh, đặt ra khả năng redundancy. Tuy nhiên chưa có feature nào bị loại chỉ dựa trên correlation; quyết định cuối sẽ dựa trên Feature Selection và clustering utility ở Phase 5–6.

# O. Điều kiện chuyển sang Phase 5

Chỉ chuyển sang **Phase 5 — Feature Selection & Feature-Set Experiments** khi:

- [ ] RFM integrity PASSED.
- [ ] Reference date đã được ghi rõ.
- [ ] `Frequency = distinct purchase days` đã được giải thích.
- [ ] Temporal feature coverage đã được biết.
- [ ] Candidate fashion-diversity features đã được tạo.
- [ ] Feature quality table đã được sinh.
- [ ] Correlation diagnostic đã được sinh.
- [ ] `feature_dictionary.csv` đã được tạo.
- [ ] `customer_features_master.parquet` đã được tạo.
- [ ] Không có feature nào bị loại chỉ vì EDA.

## Phase 5 sẽ làm gì?

1. Tạo target độc lập theo thời gian cho Filter/Embedded methods.
2. Quality filter.
3. Redundancy filter.
4. ANOVA / Mutual Information ranking.
5. Random Forest feature importance.
6. Consensus ranking.
7. Tạo Top-k candidate sets.
8. Single-addition tests.
9. Forward feature combinations.
10. Ablation experiments.
11. Chuẩn bị feature set cuối để Phase 6 chạy K-Means/GMM.